# Project 1: Deep Research Agent

Official OpenAI Agents SDK research bot pattern.

**Flow:** planner agent -> parallel search agents -> writer agent

**Official tools:** `WebSearchTool`, tracing, streaming

In [ ]:
# !pip install -q openai-agents python-dotenv

In [5]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

## 1. Imports

In [6]:
from __future__ import annotations

import asyncio
import time

from openai.types.shared.reasoning import Reasoning
from pydantic import BaseModel

from agents import Agent, ModelSettings, Runner, WebSearchTool, custom_span, gen_trace_id, trace

## 2. Planner Agent

In [7]:
PLANNER_PROMPT = (
    "You are a helpful research assistant. Given a query, come up with a set of web searches "
    "to perform to best answer the query. Output between 5 and 20 terms to query for."
)


class WebSearchItem(BaseModel):
    reason: str
    "Your reasoning for why this search is important to the query."

    query: str
    "The search term to use for the web search."


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]
    """A list of web searches to perform to best answer the query."""


planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_PROMPT,
    model="gpt-5.5",
    model_settings=ModelSettings(reasoning=Reasoning(effort="medium")),
    output_type=WebSearchPlan,
)

## 3. Search Agent

In [8]:
SEARCH_INSTRUCTIONS = (
    "You are a research assistant. Given a search term, you search the web for that term and "
    "produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 "
    "words. Capture the main points. Write succinctly, no need to have complete sentences or good "
    "grammar. This will be consumed by someone synthesizing a report, so its vital you capture the "
    "essence and ignore any fluff. Do not include any additional commentary other than the summary "
    "itself."
)


search_agent = Agent(
    name="Search agent",
    model="gpt-5.5",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[WebSearchTool()],
)

## 4. Writer Agent

In [9]:
WRITER_PROMPT = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research "
    "assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str
    """A short 2-3 sentence summary of the findings."""

    markdown_report: str
    """The final report"""

    follow_up_questions: list[str]
    """Suggested topics to research further"""


writer_agent = Agent(
    name="WriterAgent",
    instructions=WRITER_PROMPT,
    model="gpt-5-mini",
    model_settings=ModelSettings(reasoning=Reasoning(effort="medium")),
    output_type=ReportData,
)

## 5. Research Manager

In [10]:
class ResearchManager:
    async def run(self, query: str) -> ReportData:
        trace_id = gen_trace_id()
        with trace("Research trace", trace_id=trace_id):
            print(f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}")
            print("Starting research...")

            search_plan = await self._plan_searches(query)
            search_results = await self._perform_searches(search_plan)
            report = await self._write_report(query, search_results)

            print("\nReport summary\n")
            print(report.short_summary)
            return report

    async def _plan_searches(self, query: str) -> WebSearchPlan:
        print("Planning searches...")
        result = await Runner.run(planner_agent, f"Query: {query}")
        search_plan = result.final_output_as(WebSearchPlan)
        print(f"Will perform {len(search_plan.searches)} searches")
        return search_plan

    async def _perform_searches(self, search_plan: WebSearchPlan) -> list[str]:
        with custom_span("Search the web"):
            print("Searching...")
            tasks = [asyncio.create_task(self._search(item)) for item in search_plan.searches]
            results = []
            num_completed = 0
            num_succeeded = 0
            num_failed = 0

            for task in asyncio.as_completed(tasks):
                result = await task
                if result is not None:
                    results.append(result)
                    num_succeeded += 1
                else:
                    num_failed += 1

                num_completed += 1
                status = f"Searching... {num_completed}/{len(tasks)} finished"
                if num_failed:
                    status += f" ({num_succeeded} succeeded, {num_failed} failed)"
                print(status)

            print(f"Searches finished: {num_succeeded}/{len(tasks)} succeeded")
            return results

    async def _search(self, item: WebSearchItem) -> str | None:
        input_text = f"Search term: {item.query}\nReason for searching: {item.reason}"
        try:
            result = await Runner.run(search_agent, input_text)
            return str(result.final_output)
        except Exception:
            return None

    async def _write_report(self, query: str, search_results: list[str]) -> ReportData:
        print("Thinking about report...")
        input_text = f"Original query: {query}\nSummarized search results: {search_results}"
        result = Runner.run_streamed(writer_agent, input_text)

        update_messages = [
            "Thinking about report...",
            "Planning report structure...",
            "Writing outline...",
            "Creating sections...",
            "Cleaning up formatting...",
            "Finalizing report...",
            "Finishing report...",
        ]

        last_update = time.time()
        next_message = 0
        async for _ in result.stream_events():
            if time.time() - last_update > 5 and next_message < len(update_messages):
                print(update_messages[next_message])
                next_message += 1
                last_update = time.time()

        return result.final_output_as(ReportData)

## 6. Run The Research Bot

In [11]:
query = "How should engineering teams evaluate AI agents before deploying them to production?"

manager = ResearchManager()
report = await manager.run(query)

View trace: https://platform.openai.com/traces/trace?trace_id=trace_1bf4c49ba88c4caea1a7a8d5b479871c
Starting research...
Planning searches...
Will perform 15 searches
Searching...
Searching... 1/15 finished
Searching... 2/15 finished
Searching... 3/15 finished
Searching... 4/15 finished
Searching... 5/15 finished
Searching... 6/15 finished
Searching... 7/15 finished
Searching... 8/15 finished
Searching... 9/15 finished
Searching... 10/15 finished


/Users/ishandutta/miniconda3/envs/yt-agents-oai/lib/python3.11/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `literal['search']` - serialized value may not be as expected [input_value='find_in_page', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `ActionOpenPage` - serialized value may not be as expected [input_value=ActionSearch(query=None, ...safe-agent-deployments'), input_type=ActionSearch])
  PydanticSerializationUnexpectedValue(Expected `ActionFind` - serialized value may not be as expected [input_value=ActionSearch(query=None, ...safe-agent-deployments'), input_type=ActionSearch])
  return self.__pydantic_serializer__.to_python(


Searching... 11/15 finished
Searching... 12/15 finished
Searching... 13/15 finished
Searching... 14/15 finished
Searching... 15/15 finished
Searches finished: 15/15 succeeded
Thinking about report...
Thinking about report...
Planning report structure...
Writing outline...
Creating sections...
Cleaning up formatting...
Finalizing report...
Finishing report...

Report summary

Engineering teams should treat agent evaluation as a system-level, continuous discipline that tests full multi-step trajectories, tool use, state changes, and downstream outcomes—not just final answers. A robust evaluation program combines offline unit/integration tests, adversarial red-teaming, shadow/canary production checks, strong observability, human-in-the-loop escalation gates, automated CI/CD checks, and governance aligned to standards (OWASP/NIST), with clear metrics for trajectory quality, safety, and operational health.


## 7. Final Report

In [12]:
print(report.markdown_report)

# Outline

1. Executive summary
2. Why agent evaluation is different from chatbot/model evaluation
3. Core principles for evaluating AI agents
4. Pre-deployment evaluation (offline)
   - Unit and integration tests for nondeterministic agents
   - Golden sets, sample sizes, and representative inputs
   - Adversarial testing and red teaming
   - Security-focused tests (OWASP, prompt injection, tool abuse)
5. Trajectory- and process-oriented metrics
   - What to measure (task success, subgoals, tool correctness, safety)
   - Recommended metric definitions and examples
6. Human-in-the-loop (HITL) design and escalation strategy
   - Triggers, routing, reviewer UX, throughput control
7. Observability & production evaluation (online)
   - Tracing layers, logs, semantic spans, and dashboards
   - Shadow and canary strategies
   - Drift detection and failure replay
8. CI/CD, automation, and test orchestration
   - How to integrate automated evals into dev workflows
   - Regression suites and pr

## 8. Follow-Up Questions

In [13]:
for question in report.follow_up_questions:
    print("-", question)

- What domain(s) will your agent operate in (e.g., customer support, finance, healthcare, software engineering)?
- Does your agent perform side-effecting actions (DB writes, email sending, code execution, money transfer)? If so, list the critical actions.
- Which model providers and tool integrations will you use (self-hosted LLMs, OpenAI, Anthropic, Vertex AI, internal APIs)?
- What is your current CI/CD and observability stack (e.g., GitHub Actions, Jenkins, OpenTelemetry, Datadog, LangSmith)?
- What is your acceptable risk tolerance for false positives/false negatives in escalation (e.g., how often can human reviewers be involved)?
- Do you need artifacts/templates (test matrix CSV, rubric JSON, reviewer UI wireframe, runbook) customized for your environment?
